# House Prices Project

**Dataset**: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)

**Goal**: Predict the sale price of houses using statistical and machine learning methods.


---
## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add your additional imports as needed
# from sklearn...
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations
import statsmodels.formula.api as smf
# import torch

pd.set_option('display.max_columns', 100)
%matplotlib inline

---
## Load Data

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f'Training set: {train.shape[0]} rows, {train.shape[1]} columns')
print(f'Test set: {test.shape[0]} rows, {test.shape[1]} columns')

train.head()

In [ ]:
# Set a pink-themed style
pink_main = '#FF69B4'
pink_dark = '#FF1493'
pink_light = '#FFB6C1'
pink_pale = '#FFC0CB'
pink_accent = "#6D000F"

In [ ]:
# Explore the target variable
print(train['SalePrice'].describe())

fig, axes = plt.subplots(1, 1, figsize=(12, 4))
axes.hist(train['SalePrice'], bins=50, edgecolor=pink_dark, color=pink_light)
axes.set_title('SalePrice Distribution')
axes.set_xlabel('SalePrice')
plt.tight_layout()
plt.show()

print(f'Skewness: {train["SalePrice"].skew():.3f}')

The data is heavily skewed, what kind of transformation can we apply to make it more normal? 

In [ ]:
# Apply a log transformation so y' = log(price + 1) (+1 for safety/robustness)
# Then we can interpret coefficients as percentage changes instead of absolute price changes

df = train.copy()
df["LogSalePrice"] = np.log1p(df["SalePrice"])

print(df['LogSalePrice'].describe())

fig, axes = plt.subplots(1, 1, figsize=(12, 4))
axes.hist(df['LogSalePrice'], bins=50, edgecolor=pink_dark, color=pink_light)
axes.set_title('LogSalePrice Distribution')
axes.set_xlabel('LogSalePrice')
plt.tight_layout()
plt.show()

print(f'Skewness: {df["LogSalePrice"].skew():.3f}')

In [ ]:
# Explore missing values
(df.isna().sum()[df.isna().sum() > 0] / len(df)).sort_values(ascending=False) * 100

In [ ]:
# Let's clean up!
# Assume no data about pool -> no pool, and same for misc, alley and fence
df["PoolQC"] = df["PoolQC"].fillna("NA")
df["MiscFeature"] = df["MiscFeature"].fillna("NA")
df["Alley"] = df["Alley"].fillna("NA")
df["Fence"] = df["Fence"].fillna("NA")

# Assume no data about masonry area -> area = 0
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
# If masonry area is 0, veneer type should be none
df.loc[(df["MasVnrArea"] == 0) & (df["MasVnrType"].isna()), "MasVnrType"] = "None"
# For any with masonry area > 0 but veneer type NaN (4 rows) -> fill with most common veneer type
df["MasVnrType"] = df["MasVnrType"].fillna(df.loc[df["MasVnrArea"] > 0, "MasVnrType"].mode()[0])

# No fireplaces -> fireplace quality is NA
df.loc[(df["Fireplaces"] == 0) & (df["FireplaceQu"].isna()), "FireplaceQu"] = "NA"

# Same treatment for garage
df.loc[(df["GarageArea"] == 0) & (df["GarageFinish"].isna()), "GarageFinish"] = "NA"
df.loc[(df["GarageArea"] == 0) & (df["GarageQual"].isna()), "GarageQual"] = "NA"
df.loc[(df["GarageArea"] == 0) & (df["GarageType"].isna()), "GarageType"] = "NA"
df.loc[(df["GarageArea"] == 0) & (df["GarageCond"].isna()), "GarageCond"] = "NA"
# Add a separate column for garage/no garage (maybe we don't need this)
df["HasGarage"] = df["GarageYrBlt"].notna().astype(int)
# Set year built to 0 if no garage
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

# Again for basement: no basement square feet -> no basement
df.loc[(df["TotalBsmtSF"] == 0), ["BsmtFinType1", "BsmtFinType2", "BsmtExposure", "BsmtQual", "BsmtCond"]] = "NA"
# Deal with any leftover missing values
df[["BsmtFinType2", "BsmtExposure"]] = df[["BsmtFinType2", "BsmtExposure"]].fillna("NA")

# Set missing lot frontage to the neighbourhood median
df["LotFrontage"] = df["LotFrontage"].fillna(
    df.groupby("Neighborhood")["LotFrontage"].transform("median")
)

# Fill in electrical with most frequent value
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# Just because 'NA' is confusing, let's rename it to 'None'
df = df.replace("NA", "None")

In [ ]:
# Check everything is clean
(df.isna().sum()[df.isna().sum() > 0] / len(df)).sort_values(ascending=False) * 100

---
## Part 1: Classical Statistical Inference

Apply basic statistical methods to explore the data:
- **Sample mean and variance** of `SalePrice` and key features
- **Confidence intervals** for the mean SalePrice
- **Hypothesis testing** — e.g. is the mean SalePrice significantly different from \$180,000? Is the distribution of the transformed `SalePrice` normal (Shapiro-Wilk)?
- Visualize distributions and support your conclusions with plots

In [ ]:
# =============================================================================
# Part 1: Classical Statistical Inference
# =============================================================================

# --- 1.1 Sample Mean and Variance of SalePrice ---
print("=" * 60)
print("1.1 Sample Mean and Variance")
print("=" * 60)

sale_price = df["SalePrice"]
sale_price_log = df["LogSalePrice"]

n = len(sale_price)
mean_price = sale_price.mean()
var_price = sale_price.var(ddof=1)
std_price = sale_price.std(ddof=1)

mean_log = sale_price_log.mean()
var_log = sale_price_log.var(ddof=1)
std_log = sale_price_log.std(ddof=1)

print(f"SalePrice:    n={n}, mean=${mean_price:,.2f}, var={var_price:,.2f}, std=${std_price:,.2f}")
print(f"LogSalePrice: n={n}, mean={mean_log:.4f}, var={var_log:.4f}, std={std_log:.4f}")

key_features = ["GrLivArea", "TotalBsmtSF", "LotArea", "GarageArea", "YearBuilt"]
summary_stats = df[key_features].agg(["mean", "var", "std", "median", "min", "max"]).T
summary_stats.columns = ["Mean", "Variance", "Std Dev", "Median", "Min", "Max"]
print("\nKey Numerical Features - Summary Statistics:")
print(summary_stats.round(2))

In [ ]:
# --- Visualization: distributions of SalePrice and log-transformed ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sale_price, bins=50, edgecolor=pink_dark, color=pink_light, density=True)
axes[0].axvline(mean_price, color=pink_dark, linestyle='--', linewidth=2, label=f'Mean = ${mean_price:,.0f}')
axes[0].axvline(sale_price.median(), color=pink_accent, linestyle='--', linewidth=2, label=f'Median = ${sale_price.median():,.0f}')
axes[0].set_title('SalePrice Distribution', fontweight='bold')
axes[0].set_xlabel('SalePrice ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

axes[1].hist(sale_price_log, bins=50, edgecolor=pink_dark, color=pink_light, density=True)
x_norm = np.linspace(sale_price_log.min(), sale_price_log.max(), 200)
axes[1].plot(x_norm, stats.norm.pdf(x_norm, mean_log, std_log), color=pink_dark, lw=2, label='Normal fit')
axes[1].axvline(mean_log, color=pink_accent, linestyle='--', linewidth=2, label=f'Mean = {mean_log:.3f}')
axes[1].set_title('Log(SalePrice+1) Distribution', fontweight='bold')
axes[1].set_xlabel('log(SalePrice + 1)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Visualization: key features distributions ---
fig, axes = plt.subplots(1, len(key_features), figsize=(4 * len(key_features), 4))
for i, feat in enumerate(key_features):
    axes[i].hist(df[feat], bins=40, edgecolor=pink_dark, color=pink_light)
    axes[i].axvline(df[feat].mean(), color=pink_accent, linestyle='--', linewidth=1.5)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xlabel(feat)
plt.suptitle("Distributions of Key Numerical Features (dashed = mean)", y=1.02, fontweight='bold', color=pink_dark)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# --- 1.2 Confidence Intervals ---
# =============================================================================
print("=" * 60)
print("1.2 Confidence Intervals")
print("=" * 60)

alpha = 0.05

se_price = std_price / np.sqrt(n)
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
ci_price = (mean_price - t_crit * se_price, mean_price + t_crit * se_price)
print(f"\n95% CI for mean SalePrice (t-interval):")
print(f"  ({ci_price[0]:,.2f}, {ci_price[1]:,.2f})")

se_log = std_log / np.sqrt(n)
ci_log = (mean_log - t_crit * se_log, mean_log + t_crit * se_log)
print(f"\n95% CI for mean log(SalePrice+1) (t-interval):")
print(f"  ({ci_log[0]:.4f}, {ci_log[1]:.4f})")

ci_log_dollar = (np.expm1(ci_log[0]), np.expm1(ci_log[1]))
print(f"  Back-transformed to dollars: (${ci_log_dollar[0]:,.2f}, ${ci_log_dollar[1]:,.2f})")

chi2_lower = stats.chi2.ppf(alpha / 2, df=n - 1)
chi2_upper = stats.chi2.ppf(1 - alpha / 2, df=n - 1)
ci_var = ((n - 1) * var_price / chi2_upper, (n - 1) * var_price / chi2_lower)
print(f"\n95% CI for SalePrice variance (chi-squared):")
print(f"  ({ci_var[0]:,.2f}, {ci_var[1]:,.2f})")

ci_var_log = ((n - 1) * var_log / chi2_upper, (n - 1) * var_log / chi2_lower)
print(f"\n95% CI for log(SalePrice+1) variance (chi-squared):")
print(f"  ({ci_var_log[0]:,.2f}, {ci_var_log[1]:,.2f})")

print(f"\n95% Confidence Intervals for Key Feature Means:")
ci_table = []
for feat in key_features:
    feat_mean = df[feat].mean()
    feat_se = df[feat].std(ddof=1) / np.sqrt(n)
    ci_lo = feat_mean - t_crit * feat_se
    ci_hi = feat_mean + t_crit * feat_se
    ci_table.append({"Feature": feat, "Mean": feat_mean, "CI Lower": ci_lo, "CI Upper": ci_hi})
ci_df = pd.DataFrame(ci_table).set_index("Feature")
print(ci_df.round(2))

In [ ]:
# --- Visualization: CI for  ---
fig, ax = plt.subplots(figsize=(8, 3))
ax.errorbar(mean_price, 0, xerr=t_crit * se_price, fmt='o', color=pink_dark,
            capsize=10, markersize=8, linewidth=2, markerfacecolor=pink_light,
            markeredgecolor=pink_dark, ecolor=pink_main)
ax.axvline(180000, color=pink_accent, linestyle='--', linewidth=2, label='$180,000 (H₀)')
ax.set_yticks([])
ax.set_xlabel('SalePrice ($)')
ax.set_title('95% Confidence Interval for Mean SalePrice', fontweight='bold', color=pink_dark)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# --- 1.3 Hypothesis Testing ---
# =============================================================================
print("=" * 60)
print("1.3 Hypothesis Testing")
print("=" * 60)

# --- Test 1: Is mean SalePrice significantly different from $180,000? ---
mu_0 = 180000
t_stat = (mean_price - mu_0) / se_price
p_value_ttest = 2 * stats.t.sf(abs(t_stat), df=n - 1)

print(f"\nTest 1: H₀: μ = ${mu_0:,} vs H₁: μ ≠ ${mu_0:,}")
print(f"  Sample mean: ${mean_price:,.2f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value_ttest:.6f}")
print(f"  Decision at α=0.05: {'Reject H₀' if p_value_ttest < alpha else 'Fail to reject H₀'}")

t_stat_scipy, p_val_scipy = stats.ttest_1samp(sale_price, mu_0)
print(f"  (scipy verification: t={t_stat_scipy:.4f}, p={p_val_scipy:.6f})")

In [ ]:
# --- Test 2: Shapiro-Wilk normality tests ---
if n > 5000:
    sample_sw = sale_price.sample(5000, random_state=42)
else:
    sample_sw = sale_price
sw_stat, sw_p = stats.shapiro(sample_sw)
print(f"Test 2a: Shapiro-Wilk normality test on SalePrice")
print(f"  W-statistic: {sw_stat:.6f}")
print(f"  p-value: {sw_p:.6e}")
print(f"  Decision at α=0.05: {'Reject H₀ (NOT normal)' if sw_p < alpha else 'Fail to reject H₀ (normal)'}")

if n > 5000:
    sample_sw_log = sale_price_log.sample(5000, random_state=42)
else:
    sample_sw_log = sale_price_log
sw_stat_log, sw_p_log = stats.shapiro(sample_sw_log)
print(f"\nTest 2b: Shapiro-Wilk normality test on log(SalePrice+1)")
print(f"  W-statistic: {sw_stat_log:.6f}")
print(f"  p-value: {sw_p_log:.6e}")
print(f"  Decision at α=0.05: {'Reject H₀ (NOT normal)' if sw_p_log < alpha else 'Fail to reject H₀ (normal)'}")


In [ ]:
# --- Test 3: Two-sample t-test: Central Air ---
group_yes = df.loc[df["CentralAir"] == "Y", "SalePrice"]
group_no = df.loc[df["CentralAir"] == "N", "SalePrice"]
t_air, p_air = stats.ttest_ind(group_yes, group_no, equal_var=False)
print(f"Test 3: Welch's t-test — CentralAir=Y vs CentralAir=N on SalePrice")
print(f"  Mean (Yes): ${group_yes.mean():,.2f} (n={len(group_yes)})")
print(f"  Mean (No):  ${group_no.mean():,.2f} (n={len(group_no)})")
print(f"  t-statistic: {t_air:.4f}")
print(f"  p-value: {p_air:.6e}")
print(f"  Decision at α=0.05: {'Reject H₀ (significant difference)' if p_air < alpha else 'Fail to reject H₀'}")

In [ ]:
# --- Visualization: QQ plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stats.probplot(sale_price, dist="norm", plot=axes[0])
axes[0].set_title("Q-Q Plot: SalePrice", fontweight='bold', color=pink_dark)
axes[0].get_lines()[0].set_markerfacecolor(pink_light)
axes[0].get_lines()[0].set_markeredgecolor(pink_dark)
axes[0].get_lines()[0].set_markersize(3)
axes[0].get_lines()[1].set_color(pink_dark)

stats.probplot(sale_price_log, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot: log(SalePrice+1)", fontweight='bold', color=pink_dark)
axes[1].get_lines()[0].set_markerfacecolor(pink_light)
axes[1].get_lines()[0].set_markeredgecolor(pink_dark)
axes[1].get_lines()[0].set_markersize(3)
axes[1].get_lines()[1].set_color(pink_dark)

plt.tight_layout()
plt.show()

In [ ]:
# --- Visualization: Central Air boxplot ---
fig, ax = plt.subplots(figsize=(6, 5))
bp = df.boxplot(column='SalePrice', by='CentralAir', ax=ax,
                patch_artist=True, return_type='dict')
for patch in bp['SalePrice']['boxes']:
    patch.set_facecolor(pink_light)
    patch.set_edgecolor(pink_dark)
for whisker in bp['SalePrice']['whiskers']:
    whisker.set_color(pink_dark)
for cap in bp['SalePrice']['caps']:
    cap.set_color(pink_dark)
for median in bp['SalePrice']['medians']:
    median.set_color(pink_dark)
    median.set_linewidth(2)
for flier in bp['SalePrice']['fliers']:
    flier.set_markerfacecolor(pink_light)
    flier.set_markeredgecolor(pink_accent)
ax.set_title('SalePrice by Central Air Conditioning', fontweight='bold', color=pink_dark)
ax.set_xlabel('Central Air')
ax.set_ylabel('SalePrice ($)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# --- 1.4 Summary Table ---
print("=" * 60)
print("1.4 Summary of Statistical Tests")
print("=" * 60)

summary_tests = pd.DataFrame({
    "Test": [
        "One-sample t-test (μ = $180,000)",
        "Shapiro-Wilk (SalePrice)",
        "Shapiro-Wilk (log SalePrice)",
        "Welch t-test (CentralAir)"
    ],
    "Statistic": [t_stat, sw_stat, sw_stat_log, t_air],
    "p-value": [p_value_ttest, sw_p, sw_p_log, p_air],
    "Reject H₀ (α=0.05)": [
        p_value_ttest < alpha, sw_p < alpha, sw_p_log < alpha,
        p_air < alpha
    ]
})
print(summary_tests.to_string(index=False))

---
## Part 2: ANOVA — Finding Significant Features

Use ANOVA to determine which of the following **10 features** have a statistically significant effect on the transformed SalePrice. 

**Given features (10):**

| # | Feature | Levels | Description |
|---|---|---|---|
| 1 | `OverallQual` | 1–10 | Overall material and finish quality |
| 2 | `ExterQual` | Po, Fa, TA, Gd, Ex | Exterior material quality |
| 3 | `BsmtQual` | None, Po, Fa, TA, Gd, Ex | Basement height quality |
| 4 | `KitchenQual` | Po, Fa, TA, Gd, Ex | Kitchen quality |
| 5 | `FireplaceQu` | None, Po, Fa, TA, Gd, Ex | Fireplace quality |
| 6 | `CentralAir` | N, Y | Central air conditioning |
| 7 | `LotShape` | IR3, IR2, IR1, Reg | General shape of property |
| 8 | `LandSlope` | Sev, Mod, Gtl | Slope of property |
| 9 | `MoSold` | 1–12 | Month sold |
| 10 | `YrSold` | 2006–2010 | Year sold |

**Tasks:**
1. Extract these features into a dataframe and run **one-way ANOVA** on each
2. Identify which features are significant (p < 0.05)
3. Run a **two-way ANOVA** to test for interaction effects between pairs of significant features
4. Use **Tukey HSD** post-hoc tests where appropriate
5. Summarize: which features and interactions are significant?

In [ ]:
features = ["OverallQual", "ExterQual", "BsmtQual", "KitchenQual", "FireplaceQu", "CentralAir", "LotShape", "LandSlope", "MoSold", "YrSold"]

df_for_anova = df[features + ["LogSalePrice"]]
df_for_anova.head()

In [ ]:
# Run one-way ANOVA on selected features
results = []

for feature in features:
  groups = [group["LogSalePrice"].values for name, group in df_for_anova.groupby(feature)]
  f_stat, p_value = stats.f_oneway(*groups)

  results.append({
      "feature": feature,
      "f_statistic": f_stat,
      "p_value": p_value,
      "num_groups": len(groups)
  })

anova_df = pd.DataFrame(results)
anova_df = anova_df.sort_values("p_value")
anova_df["significant"] = anova_df["p_value"] < 0.05
anova_df

In [ ]:
# Show which features are significant according to one-way ANOVA
significant_features = anova_df[anova_df["significant"]]["feature"].tolist()
print(significant_features)

In [ ]:
# Run 2-way ANOVA on pairs of significant features
results = []

for f1, f2 in combinations(significant_features, 2):
    formula = f"LogSalePrice ~ C({f1}) * C({f2})"
    model = smf.ols(formula, data=df_for_anova).fit()
    anova = sm.stats.anova_lm(model, typ=2)

    results.append({
        "feature1": f1,
        "feature2": f2,
        "p_interaction": anova.loc[f"C({f1}):C({f2})", "PR(>F)"]
    })

anova2_df = pd.DataFrame(results).sort_values("p_interaction")
anova2_df["significant"] = anova2_df["p_interaction"] < 0.05
anova2_df

In [ ]:
# Check frequency of combinations using contingency table
print(pd.crosstab(df_for_anova["OverallQual"], df_for_anova["KitchenQual"]))
print(pd.crosstab(df_for_anova["FireplaceQu"], df_for_anova["LotShape"]))

In [ ]:
# Run Tukey's HSD on significant features from one-way ANOVA
for feature in significant_features:
    print("\n" + "="*50)
    print(f"Tukey HSD for {feature}")
    print("="*50)

    tukey = pairwise_tukeyhsd(
        endog=df_for_anova["LogSalePrice"],
        groups=df_for_anova[feature],
        alpha=0.05
    )

    print(tukey)

---
## Part 3: 2^k Factorial Design

Pick k binary (or binarized) factors from the significant features found in Part 2 and apply a factorial design analysis. For example you could binarize ordinal features into High/Low groups and study their joint effects.

**Tasks:**
- Select k factors (e.g. k=2 or k=3) and define High/Low levels
- Compute group means for all 2^k combinations
- Analyze main effects and interaction effects
- Visualize with interaction plots

In [ ]:
# =============================================================================
# Part 3: 2^k Factorial Design (k=3)
# =============================================================================

# Select 3 significant factors from ANOVA (Part 2) and binarize into High/Low
# Factor A: OverallQual — High (>= 7) vs Low (< 7)   [median split]
# Factor B: ExterQual   — High (Ex, Gd) vs Low (TA, Fa)
# Factor C: CentralAir  — Already binary (Y / N)

df["A_OverallQual"] = (df["OverallQual"] >= 7).map({True: "High", False: "Low"})
df["B_ExterQual"] = df["ExterQual"].map({"Ex": "High", "Gd": "High", "TA": "Low", "Fa": "Low"})
df["C_CentralAir"] = df["CentralAir"].map({"Y": "High", "N": "Low"})

print("Factor A (OverallQual — High if >= 7):")
print(df["A_OverallQual"].value_counts())
print(f"\nFactor B (ExterQual — High if Ex/Gd):")
print(df["B_ExterQual"].value_counts())
print(f"\nFactor C (CentralAir — Y=High, N=Low):")
print(df["C_CentralAir"].value_counts())

In [ ]:
# --- Group means for all 2^3 combinations ---
factorial_groups = df.groupby(["A_OverallQual", "B_ExterQual", "C_CentralAir"]).agg(
    Mean_LogPrice=("LogSalePrice", "mean"),
    Count=("LogSalePrice", "count")
).reset_index()

print("2^3 Factorial Design — Group Means:")
print(factorial_groups.to_string(index=False))

In [ ]:
# --- Visualization: group means bar chart ---
factorial_groups["Label"] = (
    factorial_groups["A_OverallQual"] + " / " +
    factorial_groups["B_ExterQual"] + " / " +
    factorial_groups["C_CentralAir"]
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(factorial_groups["Label"], factorial_groups["Mean_LogPrice"],
              color=pink_light, edgecolor=pink_dark, linewidth=1.5)
ax.set_xlabel("OverallQual / ExterQual / CentralAir", fontweight='bold')
ax.set_ylabel("Mean LogPrice")
ax.set_title("2³ Factorial Design — Mean LogPrice per Group", fontweight='bold', color=pink_dark)
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, factorial_groups["Count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'n={count}', ha='center', va='bottom', fontsize=9, color=pink_dark)

plt.tight_layout()
plt.show()

In [ ]:
model_factorial = smf.ols(
    "LogSalePrice ~ C(A_OverallQual) * C(B_ExterQual) * C(C_CentralAir)",
    data=df
).fit()

anova_table = sm.stats.anova_lm(model_factorial, typ=2)
anova_table["significant"] = anova_table["PR(>F)"] < 0.05
# Make the column names prettier
anova_table.index = anova_table.index.str.replace("C(", "", regex=False).str.replace(")", "", regex=False)

print(anova_table.index)

group_counts = df.groupby(["A_OverallQual","B_ExterQual","C_CentralAir"]).size()
print("Group counts:")
print(group_counts)

def log_effect_to_pct(x):
    return (np.exp(x) - 1) * 100

# =============================================================================
# Main Effects (log scale)
# =============================================================================
main_A = df.loc[df["A_OverallQual"]=="High", "LogSalePrice"].mean() - \
         df.loc[df["A_OverallQual"]=="Low",  "LogSalePrice"].mean()

main_B = df.loc[df["B_ExterQual"]=="High", "LogSalePrice"].mean() - \
         df.loc[df["B_ExterQual"]=="Low",  "LogSalePrice"].mean()

main_C = df.loc[df["C_CentralAir"]=="High", "LogSalePrice"].mean() - \
         df.loc[df["C_CentralAir"]=="Low",  "LogSalePrice"].mean()


# =============================================================================
# Interaction Effects (log scale)
# =============================================================================

# AB
interaction_AB = (
    (df.loc[(df["A_OverallQual"]=="High") & (df["B_ExterQual"]=="High"), "LogSalePrice"].mean() -
     df.loc[(df["A_OverallQual"]=="Low")  & (df["B_ExterQual"]=="High"), "LogSalePrice"].mean())
    -
    (df.loc[(df["A_OverallQual"]=="High") & (df["B_ExterQual"]=="Low"),  "LogSalePrice"].mean() -
     df.loc[(df["A_OverallQual"]=="Low")  & (df["B_ExterQual"]=="Low"),  "LogSalePrice"].mean())
) / 2

# AC
interaction_AC = (
    (df.loc[(df["A_OverallQual"]=="High") & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean() -
     df.loc[(df["A_OverallQual"]=="Low")  & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean())
    -
    (df.loc[(df["A_OverallQual"]=="High") & (df["C_CentralAir"]=="Low"),  "LogSalePrice"].mean() -
     df.loc[(df["A_OverallQual"]=="Low")  & (df["C_CentralAir"]=="Low"),  "LogSalePrice"].mean())
) / 2

# BC
interaction_BC = (
    (df.loc[(df["B_ExterQual"]=="High") & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean() -
     df.loc[(df["B_ExterQual"]=="Low")  & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean())
    -
    (df.loc[(df["B_ExterQual"]=="High") & (df["C_CentralAir"]=="Low"),  "LogSalePrice"].mean() -
     df.loc[(df["B_ExterQual"]=="Low")  & (df["C_CentralAir"]=="Low"),  "LogSalePrice"].mean())
) / 2

# ABC
mean_AH_BH_CH = df.loc[(df["A_OverallQual"]=="High") & (df["B_ExterQual"]=="High") & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean()
mean_AL_BH_CH = df.loc[(df["A_OverallQual"]=="Low")  & (df["B_ExterQual"]=="High") & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean()
mean_AH_BL_CH = df.loc[(df["A_OverallQual"]=="High") & (df["B_ExterQual"]=="Low")  & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean()
mean_AL_BL_CH = df.loc[(df["A_OverallQual"]=="Low")  & (df["B_ExterQual"]=="Low")  & (df["C_CentralAir"]=="High"), "LogSalePrice"].mean()

mean_AH_BH_CL = df.loc[(df["A_OverallQual"]=="High") & (df["B_ExterQual"]=="High") & (df["C_CentralAir"]=="Low"), "LogSalePrice"].mean()
mean_AL_BH_CL = df.loc[(df["A_OverallQual"]=="Low")  & (df["B_ExterQual"]=="High") & (df["C_CentralAir"]=="Low"), "LogSalePrice"].mean()
mean_AH_BL_CL = df.loc[(df["A_OverallQual"]=="High") & (df["B_ExterQual"]=="Low")  & (df["C_CentralAir"]=="Low"), "LogSalePrice"].mean()
mean_AL_BL_CL = df.loc[(df["A_OverallQual"]=="Low")  & (df["B_ExterQual"]=="Low")  & (df["C_CentralAir"]=="Low"), "LogSalePrice"].mean()

interaction_ABC = (
    mean_AH_BH_CH
    - mean_AL_BH_CH
    - mean_AH_BL_CH
    + mean_AL_BL_CH
    - mean_AH_BH_CL
    + mean_AL_BH_CL
    + mean_AH_BL_CL
    - mean_AL_BL_CL
) / 4

# =============================================================================
# Build final summary table
# =============================================================================

summary = pd.DataFrame({
    "Effect": [
        "A (OverallQual)",
        "B (ExterQual)",
        "C (CentralAir)",
        "A:B",
        "A:C",
        "B:C",
        "A:B:C"
    ],
    "Log_Effect": [
        main_A, main_B, main_C,
        interaction_AB, interaction_AC, interaction_BC, interaction_ABC
    ]
})

# Convert to %
summary["Percent_Change (%)"] = summary["Log_Effect"].apply(log_effect_to_pct)

# --- Map p-values from ANOVA ---
pval_map = {
    "A (OverallQual)": anova_table.loc["A_OverallQual", "PR(>F)"],
    "B (ExterQual)":   anova_table.loc["B_ExterQual", "PR(>F)"],
    "C (CentralAir)":  anova_table.loc["C_CentralAir", "PR(>F)"],
    "A:B":             anova_table.loc["A_OverallQual:B_ExterQual", "PR(>F)"],
    "A:C":             anova_table.loc["A_OverallQual:C_CentralAir", "PR(>F)"],
    "B:C":             anova_table.loc["B_ExterQual:C_CentralAir", "PR(>F)"],
    "A:B:C":           anova_table.loc["A_OverallQual:B_ExterQual:C_CentralAir", "PR(>F)"]
}

summary["p_value"] = summary["Effect"].map(pval_map)
summary["Significant"] = summary["p_value"] < 0.05

# --- Clean formatting ---
summary["Percent_Change (%)"] = summary["Percent_Change (%)"].round(2)
summary["p_value"] = summary["p_value"].round(4)

print("\nFinal 2^k Factorial Effects Summary:")
print(summary)

print(f"\nModel R² = {model_factorial.rsquared:.4f}")
print(f"Model Adj. R² = {model_factorial.rsquared_adj:.4f}")

In [ ]:
# --- Interaction Plot: OverallQual × ExterQual ---
fig, ax = plt.subplots(figsize=(8, 5))

for level, color, marker in [("High", pink_dark, 'o'), ("Low", pink_accent, 's')]:
    means = df[df["B_ExterQual"] == level].groupby("A_OverallQual")["LogSalePrice"].mean()
    ax.plot(means.index, means.values, marker=marker, color=color,
            linewidth=2, markersize=8, label=f'ExterQual = {level}')

ax.set_xlabel("OverallQual Level", fontweight='bold')
ax.set_ylabel("Mean LogSalePrice")
ax.set_title("Interaction Plot: OverallQual × ExterQual", fontweight='bold', color=pink_dark)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Interaction Plot: OverallQual × CentralAir ---
fig, ax = plt.subplots(figsize=(8, 5))

for level, color, marker in [("High", pink_dark, 'o'), ("Low", pink_accent, 's')]:
    means = df[df["C_CentralAir"] == level].groupby("A_OverallQual")["LogSalePrice"].mean()
    ax.plot(means.index, means.values, marker=marker, color=color,
            linewidth=2, markersize=8, label=f'CentralAir = {level}')

ax.set_xlabel("OverallQual Level", fontweight='bold')
ax.set_ylabel("Mean LogSalePrice")
ax.set_title("Interaction Plot: OverallQual × CentralAir", fontweight='bold', color=pink_dark)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Interaction Plot: ExterQual × CentralAir ---
fig, ax = plt.subplots(figsize=(8, 5))

for level, color, marker in [("High", pink_dark, 'o'), ("Low", pink_accent, 's')]:
    means = df[df["C_CentralAir"] == level].groupby("B_ExterQual")["LogSalePrice"].mean()
    ax.plot(means.index, means.values, marker=marker, color=color,
            linewidth=2, markersize=8, label=f'CentralAir = {level}')

ax.set_xlabel("ExterQual Level", fontweight='bold')
ax.set_ylabel("Mean LogSalePrice")
ax.set_title("Interaction Plot: ExterQual × CentralAir", fontweight='bold', color=pink_dark)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Summary of all effects ---
print("=" * 60)
print("Summary of Factorial Effects")
print("=" * 60)

effects_summary = pd.DataFrame({
    "Effect": ["A (OverallQual)", "B (ExterQual)", "C (CentralAir)",
               "A × B", "A × C", "B × C"],
    "Type": ["Main", "Main", "Main", "Interaction", "Interaction", "Interaction"],
    "Effect Size ($)": [main_A, main_B, main_C, interaction_AB, interaction_AC, interaction_BC]
})
print(effects_summary.to_string(index=False))

# Bar chart of effects
fig, ax = plt.subplots(figsize=(10, 5))
colors = [pink_dark]*3 + [pink_light]*3
bars = ax.bar(effects_summary["Effect"], effects_summary["Effect Size ($)"],
              color=colors, edgecolor=pink_dark, linewidth=1.5)
ax.set_ylabel("Effect Size ($)")
ax.set_title("Main Effects and Interaction Effects", fontweight='bold', color=pink_dark)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Part 4: Parametric Regression

Build a regression model using only the **significant ordinal features** identified by ANOVA (Part 2) plus the **2 numerical features**: `GrLivArea` and `TotalBsmtSF`.

**Tasks:**
- Encode ordinal features numerically (map quality levels to integers)
- Fit a linear regression model (OLS)
- Analyze the model: R², coefficient significance, residual plots
- Optionally try regularized regression (Ridge, Lasso) and compare
- Apply ANOVA on the regression model to assess factor contributions

---
## Part 5: Non-Parametric Model (Neural Network)

Build a neural network regression model using **all** available features to predict SalePrice. This is also the model that produces your `submission.csv` for Kaggle scoring.

**Tasks:**
- Preprocess all features: handle missing values, encode categoricals, scale numerics
- Build and train a neural network (e.g. `sklearn.neural_network.MLPRegressor` or PyTorch)
- Evaluate on training data (RMSE, R²) and analyze residuals
- Generate predictions for the test set and save as `submission.csv`

**Important:** The Kaggle RMSE score is evaluated on the predictions from this model.